In [1]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import scanpy as sc
import anndata as ad
import bbknn
from sklearn.decomposition import PCA
import numpy as np
import harmonypy as hm

/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
train_id=['CRC0327','CRC0542','CRC0322']
tratt_cercato=['NT','NT72h']
train=pd.DataFrame()
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    if sample_name in train_id and trattamento in tratt_cercato:
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])



In [3]:
import sys, importlib
sys.path.append("/percorso/del/tuo/modulo")  # solo se non è già nel path

import func            # oppure: import sc_pipeline
importlib.reload(func) # forza il reload dopo aver salvato func.py


<module 'func' from '/mnt/cold1/snaketree/prj/scRNA/dataset/Graph_NN/notebook/func.py'>

In [4]:
from func import *

df=train
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample"), sep=":", on_duplicate="first")

# 2) AnnData + uid + raw
adata_full = make_anndata_from_df(df_clean, set_raw=True)

# 3) cell cycle
score_cell_cycle(adata_full)

# 4) HVG + PCA
select_hvg_cell_ranger(adata_full, n_top_genes=1500, batch_key="sample", subset=True)
scale_and_pca(adata_full, n_comps=42, max_value=10, random_state=42)
adata_base = adata_full

GRID = {
    "n_pcs": [20,40],
    "umap_n_neighbors": 15,
    "bbknn_neighbors_within_batch": [], #"bbknn_neighbors_within_batch": [3, 5],
    "bbknn_trim": [],#"bbknn_trim": [None, 10]
    "harmony_theta": [2.0, 4.0],
    "harmony_lambda": [2.0],
    "leiden_res": [0.3,0.4,0.5,0.6],
    "louvain_res": [0.3,0.4,0.5,0.6]
}
GENE_PANEL = ["ATOH1","DLL1","DLL4","GFI1","AREG","HES1","HES5","JAG2","NOTCH1","NOTCH2","NOTCH3",
              "OLFM4","LEF1","APCDD1","WNT6","NEUROG3","NEUROD1","KRT20","NEURL1","LGR5"]

df_results = run_grid(
    adata_base=adata_base,
    grid=GRID,
    out_root="./results_grid",
    batch_key="sample",
    gene_panel=GENE_PANEL,
    random_state=42
)


TypeError: run_harmony() got an unexpected keyword argument 'lambda'

In [ ]:
# === import dal tuo modulo ===
from func import *

# 1) pulizia nomi gene
df = train  # il tuo DataFrame (righe=celle; colonne=geni + 'cell_id' + 'sample')
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample"), sep=":", on_duplicate="first")

# 2) AnnData + uid + raw
adata_full = make_anndata_from_df(df_clean, set_raw=True)  # crea obs['uid'] e lo usa come index

# 3) cell cycle a priori (S_score, G2M_score, phase)
score_cell_cycle(adata_full)


adata_base = make_base_cc_regressed(adata_full, n_hvg=1500, batch_key="sample", n_pcs=60, rng=42)


GRID = {
    "n_pcs": [20,40],
    "umap_n_neighbors": 15,
    "bbknn_neighbors_within_batch": [], #"bbknn_neighbors_within_batch": [3, 5],
    "bbknn_trim": [],#"bbknn_trim": [None, 100]
    "harmony_theta": [2.0, 4.0],
    "harmony_lambda": [2.0],
    "leiden_res": [0.3,0.4,0.5,0.6],
    "louvain_res": [0.3,0.4,0.5,0.6]
}

GENE_PANEL = [
    "ATOH1","DLL1","DLL4","GFI1","AREG","HES1","HES5","JAG2","NOTCH1","NOTCH2","NOTCH3",
    "OLFM4","LEF1","APCDD1","WNT6","NEUROG3","NEUROD1","KRT20","NEURL1","LGR5"
]

# 6) run grid (salva UMAP pannello, UMAP phase, UMAP sample, UMAP per clustering, markers top-25, metriche)
df_results = run_grid(
    adata_base=adata_base,
    grid=GRID,
    out_root="./results_grid_cc",
    batch_key="sample",
    gene_panel=GENE_PANEL,
    random_state=42
)

df_results.head()


In [ ]:
df=train
import pandas as pd
import anndata as ad
import scanpy as sc

# --- INPUT ---
# df: DataFrame con righe = cellule, colonne = geni + ['cell_id','sample']

meta_cols = ['cell_id', 'sample']
gene_cols = [c for c in df.columns if c not in meta_cols]

adata = ad.AnnData(
    X=df[gene_cols].to_numpy(),
    obs=pd.DataFrame({
        'cell_id': df['cell_id'].astype(str).values,
        'sample': df['sample'].astype(str).values
    })
)
# crea uid univoco per cella
adata.obs['uid'] = adata.obs['sample'] + '|' + adata.obs['cell_id']
adata.obs_names = adata.obs['uid'].values
adata.var_names = pd.Index(gene_cols)


adata.layers['log2cpm'] = adata.X.copy()

In [ ]:
import scanpy as sc
import pandas as pd
import re

SEP="|"
if "uid" not in adata_full.obs.columns:
    adata_full.obs["uid"] = adata_full.obs["sample"].astype(str) + SEP + adata_full.obs["cell_id"].astype(str)
adata_full.obs_names = adata_full.obs["uid"].astype(str)

# --- imposta raw = full per tenere tutti i geni anche dopo HVG -------
adata_full.raw = adata_full

# --- Gene list per fase del ciclo (umano, set Seurat/Tirosh) ----------
S_GENES = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
G2M_GENES = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]

# helper per matchare geni (case-insensitive)
def match_genes_names(var_names, genes):
    vset = set(var_names)
    present = [g for g in genes if g in vset]
    if present: return present
    vm = {g.upper(): g for g in var_names}
    return [vm[g.upper()] for g in genes if g.upper() in vm]

s_present  = match_genes_names(adata_full.var_names, S_GENES)
g_present  = match_genes_names(adata_full.var_names, G2M_GENES)

sc.tl.score_genes_cell_cycle(adata_full, s_genes=s_present, g2m_genes=g_present)
# Ora adata_full.obs ha: 'S_score', 'G2M_score', 'phase'


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

HVG_LIST = [1000, 1500, 2000, 3000, 4000]   
TARGET   = 0.85
MAX_PCS  = 100       
RNG      = 42

def pcs_needed_for_target(adata_in, target=0.90, max_n_comps=200, random_state=42):
    """Ritorna (#PC per raggiungere 90% max_n_comps)."""
    sc.tl.pca(adata_in, n_comps=max_n_comps, svd_solver="arpack", random_state=random_state)
    cum = adata_in.uns["pca"]["variance_ratio"].cumsum()
    pcs = int(np.searchsorted(cum, target) + 1) if cum[-1] >= target else None
    return pcs, float(cum[-1])

results = []
results_={}
for n_top in HVG_LIST:
    ad_tmp = adata.copy()  # usa il tuo oggetto 'full'
    sc.pp.highly_variable_genes(
        ad_tmp, flavor="cell_ranger", n_top_genes=n_top, batch_key="sample", subset=True
    )
    sc.pp.scale(ad_tmp, max_value=10)
    pcs90, cum_last = pcs_needed_for_target(ad_tmp, target=TARGET, max_n_comps=MAX_PCS, random_state=RNG)
    results.append({"hvg": n_top, "pcs_for_90": pcs90, "cum_with_max": cum_last})
    evr = ad_tmp.uns['pca']['variance_ratio']
    results_[n_top] = evr
# --- plot: quante PC servono ---
x = [r["hvg"] for r in results]
y = [r["pcs_for_90"] if r["pcs_for_90"] is not None else MAX_PCS for r in results]

plt.figure(figsize=(7,5))
plt.bar(range(len(x)), y)
plt.xticks(range(len(x)), x)
plt.ylabel("PC necessarie per raggiungere il 90%")
plt.xlabel("Numero di HVG (cell_ranger)")
plt.title("Quante PC servono per il 90% di varianza (per ogni HVG)")
# etichette sopra le barre + nota se non raggiunge il target
for i, r in enumerate(results):
    label = str(r["pcs_for_90"]) if r["pcs_for_90"] is not None else f">={MAX_PCS}\n(cum={r['cum_with_max']:.2f})"
    plt.text(i, y[i] + 1, label, ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

# (opzionale) stampa tabella riassuntiva
for r in results:
    print(f"HVG={r['hvg']}: PC@90% = {r['pcs_for_90']}, cumulativa con {MAX_PCS} PC = {r['cum_with_max']:.3f}")



In [ ]:
plt.figure(figsize=(7,5))
for n_top, evr in results_.items():
    cumvar = evr.cumsum()
    plt.plot(range(1, len(cumvar)+1), cumvar, label=f"HVG={n_top}")

plt.axhline(0.9, ls="--", c="grey")   # soglia 90% varianza spiegata
plt.xlabel("Numero di componenti principali")
plt.ylabel("Varianza spiegata cumulativa")
plt.legend()
plt.title("Confronto varianza spiegata con diversi HVG")
plt.tight_layout()
plt.show()
for n_top, evr in results_.items():
    cumvar = evr.cumsum()
    pc90 = (cumvar >= 0.9).argmax() + 1  
    print(f"HVG={n_top}: 90% varianza spiegata con {pc90} PC")

In [ ]:

# --- STEP 1: HVG selection ---
# cell_ranger+ batch-aware: prende i top HVG per ciascun sample e fa l'unione
sc.pp.highly_variable_genes(
    adata,
    flavor='cell_ranger',
    n_top_genes=1500,        
    subset=True              
)


print(f"HVG selezionati: {adata.n_vars}")


In [ ]:
adata_full.raw()

In [ ]:
adata_base_cc = make_base_cc_regressed(adata_full, n_hvg=1000)
# Poi lanci la tua grid sul base CORRETTO:
df_cc = run_grid(adata_base=adata_base_cc, grid=GRID, out_root="./results_grid_cc",
                 batch_key="sample", gene_panel=GENE_PANEL, random_state=42)